In [ ]:
from __future__ import annotations

from pathlib import Path
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import cdsapi


plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.max_columns = 120

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "era5"
INTERIM_DIR = DATA_DIR / "interim" / "era5"
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

YEAR = "2023"

# Keep the notebook default intentionally small. Expand only after validating credentials,
# storage, and the downstream feature pipeline.
REQUEST_MONTHS = ["01"]
REQUEST_DAYS = [f"{day:02d}" for day in range(1, 8)]
REQUEST_HOURS = [f"{hour:02d}:00" for hour in range(24)]
MAX_SAFE_TIME_STEPS = 31 * 24
RUN_ERA5_DOWNLOAD = True

ERA5_VARIABLES = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "total_precipitation",
    "surface_solar_radiation_downwards",
    "total_cloud_cover",
]

REQUEST_LABEL = f"{YEAR}_{REQUEST_MONTHS[0]}{REQUEST_DAYS[0]}_{REQUEST_MONTHS[-1]}{REQUEST_DAYS[-1]}"
ERA5_ARCHIVE_FILE = RAW_DIR / f"era5_single_levels_pjm_box_hourly_{REQUEST_LABEL}.zip"
ERA5_EXTRACT_DIR = RAW_DIR / f"era5_single_levels_pjm_box_hourly_{REQUEST_LABEL}"
FEATURE_FILE = INTERIM_DIR / f"era5_pjm_box_hourly_features_{REQUEST_LABEL}.parquet"

# Broad bounding box for PJM-adjacent weather exposure: north, west, south, east.
# This is intentionally approximate for initial exploration. Replace with load-weighted zones later.
PJM_AREA = [43.5, -91.0, 35.0, -73.0]

ERA5_ARCHIVE_FILE, ERA5_EXTRACT_DIR, FEATURE_FILE

# ERA5 Hourly Weather Exploration for PJM

Purpose: download or load ERA5 hourly single-level weather data for a broad PJM footprint and create first weather diagnostics relevant to electricity demand.

Research guardrails:
- ERA5 reanalysis is excellent for historical analysis, but it is not an operational forecast available before real time.
- Use ERA5 carefully in forecasting experiments: realized weather at forecast target time is leakage for many horizons.
- Preserve the raw NetCDF file and create explicit derived tabular features.

Primary source: Copernicus Climate Data Store, ERA5 hourly data on single levels.

## Download ERA5, If Needed

This cell uses `cdsapi`. It requires CDS credentials configured locally, usually through `~/.cdsapirc`.

The default request is intentionally limited to one week. Full-year hourly regional NetCDF pulls can be large and should be handled later by month/season batches with explicit storage checks.

Variables requested:
- 2m temperature
- 2m dewpoint temperature
- 10m u/v wind components
- total precipitation
- surface solar radiation downwards
- total cloud cover

In [ ]:
def download_era5_single_levels(
    target: Path,
    year: str,
    area: list[float],
    months: list[str],
    days: list[str],
    hours: list[str],
    variables: list[str],
    allow_large_request: bool = False,
) -> None:
    requested_time_steps = len(months) * len(days) * len(hours)
    if requested_time_steps > MAX_SAFE_TIME_STEPS and not allow_large_request:
        raise ValueError(
            f"ERA5 request has {requested_time_steps:,} time steps. "
            f"The notebook guardrail is {MAX_SAFE_TIME_STEPS:,}. "
            "Download larger ranges in explicit monthly/seasonal batches after validating storage."
        )
    client = cdsapi.Client()
    request = {
        "product_type": ["reanalysis"],
        "variable": variables,
        "year": [year],
        "month": months,
        "day": days,
        "time": hours,
        "area": area,
        "data_format": "netcdf",
        "download_format": "unarchived",
    }
    client.retrieve("reanalysis-era5-single-levels", request, str(target))


if not ERA5_ARCHIVE_FILE.exists() and not ERA5_EXTRACT_DIR.exists():
    print(f"ERA5 archive not found: {ERA5_ARCHIVE_FILE}")
    print(
        "Default request is one week. Set RUN_ERA5_DOWNLOAD = True in the first cell to download it."
    )
    if RUN_ERA5_DOWNLOAD:
        download_era5_single_levels(
            ERA5_ARCHIVE_FILE,
            YEAR,
            PJM_AREA,
            REQUEST_MONTHS,
            REQUEST_DAYS,
            REQUEST_HOURS,
            ERA5_VARIABLES,
        )
else:
    print("Using cached ERA5 download or extracted files.")

## Load NetCDF and Inspect Metadata

In [ ]:
def prepare_era5_netcdf_files() -> list[Path]:
    ERA5_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    archive = ERA5_ARCHIVE_FILE

    if archive.exists() and zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(ERA5_EXTRACT_DIR)

    netcdf_files = sorted(ERA5_EXTRACT_DIR.glob("*.nc"))
    if not netcdf_files:
        raise FileNotFoundError(
            f"Place ERA5 NetCDF files in {ERA5_EXTRACT_DIR}, or set RUN_ERA5_DOWNLOAD = True in the first cell."
        )
    return netcdf_files


netcdf_files = prepare_era5_netcdf_files()
datasets = [xr.open_dataset(path, engine="netcdf4") for path in netcdf_files]
ds = xr.merge(datasets, compat="override") if len(datasets) > 1 else datasets[0]
ds

In [ ]:
time_dim = "valid_time" if "valid_time" in ds.coords else "time"
lat_dim = "latitude"
lon_dim = "longitude"

print("time dimension:", time_dim)
print("variables:", list(ds.data_vars))
print(
    "time range:",
    pd.to_datetime(ds[time_dim].values.min()),
    pd.to_datetime(ds[time_dim].values.max()),
)

## Area-Average Weather Features

This first pass uses a simple spatial mean over the PJM bounding box. Later work should replace this with population-weighted or load-zone-weighted weather exposure.

In [ ]:
rename_map = {
    "t2m": "temperature_2m_k",
    "2m_temperature": "temperature_2m_k",
    "d2m": "dewpoint_2m_k",
    "2m_dewpoint_temperature": "dewpoint_2m_k",
    "u10": "wind_u_10m_ms",
    "10m_u_component_of_wind": "wind_u_10m_ms",
    "v10": "wind_v_10m_ms",
    "10m_v_component_of_wind": "wind_v_10m_ms",
    "tp": "total_precipitation_m",
    "total_precipitation": "total_precipitation_m",
    "ssrd": "surface_solar_radiation_downwards_j_m2",
    "surface_solar_radiation_downwards": "surface_solar_radiation_downwards_j_m2",
    "tcc": "total_cloud_cover_fraction",
    "total_cloud_cover": "total_cloud_cover_fraction",
}

available = {k: v for k, v in rename_map.items() if k in ds.data_vars}
if not available:
    raise ValueError(
        "No expected ERA5 short-name variables found. Inspect ds.data_vars and update rename_map."
    )

weather_grid = ds[list(available)].rename(available)
weather_area_mean = weather_grid.mean(dim=[lat_dim, lon_dim], skipna=True)
weather_features = (
    weather_area_mean.to_dataframe()
    .reset_index()
    .rename(columns={time_dim: "timestamp_utc"})
)
weather_features["timestamp_utc"] = pd.to_datetime(
    weather_features["timestamp_utc"], utc=True
)

if "temperature_2m_k" in weather_features:
    weather_features["temperature_2m_c"] = weather_features["temperature_2m_k"] - 273.15
    weather_features["temperature_2m_f"] = (
        weather_features["temperature_2m_c"] * 9 / 5 + 32
    )
if "dewpoint_2m_k" in weather_features:
    weather_features["dewpoint_2m_c"] = weather_features["dewpoint_2m_k"] - 273.15
if {"wind_u_10m_ms", "wind_v_10m_ms"}.issubset(weather_features.columns):
    weather_features["wind_speed_10m_ms"] = np.sqrt(
        weather_features["wind_u_10m_ms"] ** 2 + weather_features["wind_v_10m_ms"] ** 2
    )

weather_features = weather_features.sort_values("timestamp_utc").reset_index(drop=True)
weather_features.to_parquet(FEATURE_FILE, index=False)
weather_features.head(), weather_features.shape

## Weather Integrity Checks

In [ ]:
expected = pd.date_range(
    weather_features["timestamp_utc"].min(),
    weather_features["timestamp_utc"].max(),
    freq="h",
    tz="UTC",
)
missing_hours = expected.difference(pd.DatetimeIndex(weather_features["timestamp_utc"]))

pd.Series(
    {
        "start_utc": weather_features["timestamp_utc"].min(),
        "end_utc": weather_features["timestamp_utc"].max(),
        "rows": len(weather_features),
        "missing_hours": len(missing_hours),
        "duplicate_hours": weather_features["timestamp_utc"].duplicated().sum(),
    }
)

In [ ]:
weather_features.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

## First Weather Diagnostics

In [ ]:
plot_cols = [
    col
    for col in [
        "temperature_2m_f",
        "dewpoint_2m_c",
        "wind_speed_10m_ms",
        "total_cloud_cover_fraction",
    ]
    if col in weather_features
]

axes = weather_features.set_index("timestamp_utc")[plot_cols].plot(
    subplots=True, figsize=(14, 2.5 * len(plot_cols)), linewidth=0.8
)
plt.tight_layout()
plt.show()

In [ ]:
if "temperature_2m_f" in weather_features:
    fig, ax = plt.subplots(figsize=(8, 4))
    weather_features["temperature_2m_f"].hist(bins=60, ax=ax)
    ax.set_title("Area-mean PJM-box hourly temperature")
    ax.set_xlabel("Temperature (F)")
    ax.set_ylabel("Hour count")
    plt.show()

## Next Research Questions

- How much do simple bounding-box averages differ from population-weighted weather exposure?
- Which weather variables matter most by season: temperature, dew point, wind chill, cloud cover, solar radiation?
- What is the appropriate non-leaky weather input for the chosen forecast horizon: realized weather, lagged weather, or operational forecasts?